In [0]:
from pyspark.sql import functions as F

catalog = 'olist'
base = f'/Volumes/{catalog}/bronze/landing'   # same base as your bronze notebook
tables = ['orders','order_items','payments','reviews','customers','products','sellers','category_translation']


In [0]:
# STEP 1: wipe old tables and Auto Loader state (your CSV files in landing are NOT touched)
for t in tables:
    spark.sql(f'DROP TABLE IF EXISTS {catalog}.bronze.{t}')
    for p in [f'metadata/_chk/{t}', f'metadata/_schema/{t}', f'_chk/{t}', f'_schema/{t}']:
        dbutils.fs.rm(f'{base}/{p}', True)

In [0]:
# STEP 2: reload every table with your ingestion code
def ingest(t):
    print(f"Starting Auto Loader ingestion for table: {t}...")
    reader = (spark.readStream.format('cloudFiles')
      .option('cloudFiles.format', 'csv')
      .option('header', 'true')
      .option('cloudFiles.inferColumnTypes', 'true')
      .option('cloudFiles.schemaLocation', f'{base}/metadata/_schema/{t}'))

    if t == 'reviews':
        reader = (reader.option('multiLine', 'true')
                        .option('quote', '"')
                        .option('escape', '"'))

    (reader.load(f'{base}/{t}/')
      .withColumn('_ingested_at', F.current_timestamp())
      .withColumn('_source_file', F.col('_metadata.file_path'))
      .writeStream
      .format('delta')
      .option('checkpointLocation', f'{base}/metadata/_chk/{t}')
      .trigger(availableNow=True)
      .toTable(f'{catalog}.bronze.{t}'))

for t in tables:
    ingest(t)

In [0]:
expected = {'orders':99441,'order_items':112650,'payments':103886,'reviews':99224,
            'customers':99441,'products':32951,'sellers':3095,'category_translation':71}
for t, n in expected.items():
    c = spark.table(f'{catalog}.bronze.{t}').count()
    print(t, c, 'OK' if c == n else f'MISMATCH (expected {n})')